# Feature Engineering (Experiment): Using Pre-computed Embeddings

This is an experimental variant of the original feature engineering notebook. Instead of generating new embeddings with `all-MiniLM-L6-v2`, we directly use the **3072-dimensional pre-computed embeddings** already present in the `DeepMostInnovations/saas-sales-conversations` dataset, reduce them with PCA, and feed them into the same modeling pipeline.

The goal is to compare whether the higher-dimensional native embeddings carry stronger semantic signal than the 384-dim MiniLM embeddings used in Phase 1.

In [1]:
# 1. Load and prepare data
import pandas as pd
import numpy as np
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

# Load dataset
dataset = load_dataset("DeepMostInnovations/saas-sales-conversations", split="train")

# Convert to pandas DataFrame for feature engineering
df = dataset.to_pandas().sample(n=8000, random_state=42)

# Keep pre-computed embeddings this time. Only drop JSON/identifier columns.
cols_to_drop = ['scenario', 'conversation', 'probability_trajectory', 'conversation_id', 'company_id', 'company_name']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

embedding_cols = [col for col in df.columns if col.startswith('embedding_')]
print(f"Initial shape: {df.shape}")
print(f"Number of pre-computed embedding dimensions: {len(embedding_cols)}")

README.md: 0.00B [00:00, ?B/s]

cleaned_custom_dataset.csv:   0%|          | 0.00/7.17G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Initial shape: (8000, 3082)
Number of pre-computed embedding dimensions: 3072


In [2]:
# 2. Basic feature preparation
categorical_cols = ['product_name', 'product_type', 'conversation_style', 'conversation_flow', 'communication_channel']

# One-Hot Encoding for small categories
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [3]:
# 3. Text Feature Engineering (CORE PART)
# Basic statistical text features
df['text_length'] = df['full_text'].astype(str).apply(len)
df['word_count'] = df['full_text'].astype(str).apply(lambda x: len(x.split()))
df['average_word_length'] = df['text_length'] / (df['word_count'] + 1)

# Extract pre-computed embeddings matrix directly from dataset
embedding_matrix = df[embedding_cols].values.astype(np.float32)
print(f"Pre-computed embedding matrix shape: {embedding_matrix.shape}")

# Drop the 3072 raw embedding columns from df; PCA-reduced versions will be added later
df = df.drop(columns=embedding_cols)

Pre-computed embedding matrix shape: (8000, 3072)


In [4]:
# 4. Interaction Features
df['engagement_x_length'] = df['customer_engagement'] * df['conversation_length']
df['engagement_per_turn'] = df['customer_engagement'] / (df['conversation_length'] + 1)
df['text_per_turn'] = df['text_length'] / (df['conversation_length'] + 1)

In [5]:
# 5. Distribution-based Transformations
from sklearn.preprocessing import StandardScaler

skewed_features = ['text_length', 'word_count', 'conversation_length']
for feat in skewed_features:
    df[f'{feat}_log'] = np.log1p(df[feat])

features_to_scale = ['customer_engagement', 'sales_effectiveness', 'average_word_length',
                     'engagement_x_length', 'engagement_per_turn', 'text_per_turn'] + [f'{f}_log' for f in skewed_features]

scaler = StandardScaler()
df[features_to_scale] = scaler.fit_transform(df[features_to_scale])

In [6]:
# 6. Dimensionality Reduction on Pre-computed Embeddings
from sklearn.decomposition import PCA

n_components = 20
pca = PCA(n_components=n_components, random_state=42)
reduced_embeddings = pca.fit_transform(embedding_matrix)

print(f"Explained variance ratio by {n_components} components: {sum(pca.explained_variance_ratio_):.4f}")

for i in range(n_components):
    df[f'emb_pca_{i}'] = reduced_embeddings[:, i]

df = df.drop(columns=['full_text'])

Explained variance ratio by 20 components: 0.6007


In [7]:
# 7. Feature Selection (lightweight)
from sklearn.ensemble import RandomForestClassifier

X = df.drop(columns=['outcome'])
y = df['outcome']

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top 10 features:\n", importances.head(10))

Top 10 features:
 sales_effectiveness        0.331450
customer_engagement        0.221531
engagement_per_turn        0.137917
engagement_x_length        0.083128
emb_pca_17                 0.010556
emb_pca_19                 0.009539
emb_pca_15                 0.008151
emb_pca_13                 0.007921
conversation_length_log    0.007684
emb_pca_8                  0.007407
dtype: float64


In [8]:
# 8. Final Feature Set
print(f"Final feature matrix shape: {df.shape}")

# Save for downstream modeling
df.to_parquet("saas_features_pretrained.parquet", index=False)
print("File saved successfully")

df.head()

Final feature matrix shape: (8000, 96)
File saved successfully


,outcome,conversation_length,customer_engagement,sales_effectiveness,product_name_AR Renewable Insights,product_name_AutoLogistics Pro,product_name_BioSync,product_name_DataFlow Pro,product_name_DevOps Automation Suite,product_name_EcoSecure Manager,...,emb_pca_10,emb_pca_11,emb_pca_12,emb_pca_13,emb_pca_14,emb_pca_15,emb_pca_16,emb_pca_17,emb_pca_18,emb_pca_19
75721,1,12,0.492069,1.193471,False,False,False,False,False,False,...,-0.140102,-0.131805,-0.126564,-0.019149,-0.043225,0.030330,0.079839,0.029264,0.053344,0.073141
80184,1,11,0.919787,1.510406,False,False,False,False,False,False,...,-0.038922,-0.023879,0.003733,-0.041790,-0.147418,0.092356,0.043477,0.027836,0.058949,-0.054713
19864,1,10,0.492069,1.193471,False,False,False,False,False,False,...,-0.063550,0.069125,0.040028,0.064986,0.084621,0.026569,0.101662,-0.047199,0.070297,-0.051567
76699,0,13,-0.363368,-0.708134,False,False,False,False,False,False,...,-0.098147,-0.058958,-0.147658,0.022174,-0.074922,-0.017612,-0.001292,0.009306,0.011891,-0.119218
92991,1,9,0.492069,1.193471,False,True,False,False,False,False,...,0.181518,-0.025206,-0.017527,-0.102812,0.108841,0.033145,-0.009860,-0.088956,0.075106,0.073807


### Conclusion

- **Pre-computed Embeddings**: We directly used the 3072-dim embeddings shipped with the dataset (generated by a larger model than MiniLM). PCA reduces them to 20 components for apples-to-apples comparison with the original notebook.
- **Interaction, Transformation, Scaling**: Identical to the original pipeline, ensuring any performance delta is attributable to the embedding source alone.
- **Expected Outcome**: If the native embeddings are richer, the semantic-only and hybrid models should show improvement over the MiniLM-based Phase 1 results.